# Supplementary Figure 8 — sand extraction reduction by CE measure

Cumulative-measure wedge plot (BAU -> CE) for **sand**, 30 economies per page (5 pages, 142 economies total).

Each rung adds one CE lever; the wedge between two rungs is that measure's extraction saving. Trajectories are the **per-year median over n_sim=10,000** (a single draw is jagged; the median is the smooth central estimate). Endpoints reuse the production `BAU`/`CE` scenarios; the 5 intermediate rungs (`wedge_MIU…wedge_SS`) come from `1-IMAGINE-Concrete/run-wedge-scenarios.py`.

> Run in the activated `wty` kernel: this env's numpy LAPACK crashes (`0xc06d007f`) if Python is started without the env activated.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / '1-IMAGINE-Concrete'))

import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

from IMAGINE_Concrete import RESULTS
MF = RESULTS / 'material-flows'

In [2]:
ROOT    = Path('..')
DATA    = ROOT / 'data'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'

In [3]:
economies_142 = pd.read_csv(DATA / 'production-142.csv', index_col=0).columns.tolist()

# cumulative-measure ladder rung -> on-disk scenario folder (all n_sim=10,000):
#   BAU/CE reuse the production scenarios; the 5 intermediates are wedge_*.
FOLDER = {
    'BAU': 'BAU', 'MIU': 'wedge_MIU', 'LE': 'wedge_LE', 'RE': 'wedge_RE',
    'RC': 'wedge_RC', 'SS': 'wedge_SS', 'CE': 'CE',
}
RUNGS = ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CE']
HIST = 'historical'        # production 10,000-run historical
HIST_START = 1931          # historical npz columns span 1931..2022
MID, END = 2022, 2050      # projection npz columns span 2023..2050

def _fut(rung, economy, key):
    return np.median(np.load(MF / economy / FOLDER[rung] / f'{key}.npz')['data'], axis=0)

def _hist(economy, key):
    return np.median(np.load(MF / economy / HIST / f'{key}.npz')['data'], axis=0)

In [4]:
def draw_ME_wedge_for_a_material(material, country, axx):
    key = f'{material}_extraction'        # gravel_/sand_/limestone_extraction

    # per-year median future extraction for each ladder rung (2023..2050)
    E = {r: _fut(r, country, key) for r in RUNGS}
    bau = E['BAU']

    # historical extraction (2010..2022)
    hist = _hist(country, key)
    hist_2010_2022 = hist[2010 - HIST_START: 2022 - HIST_START + 1]
    seam = hist[2022 - HIST_START]

    # y-axis unit picked from the BAU peak (values are in kt)
    the_max_ = bau.max() * 1.3 / 1e6
    if the_max_ <= 1:
        yaxis_scale = 'Mt'
        if the_max_ <= 0.001:
            yaxis_scale = 'kt'
    else:
        yaxis_scale = 'Gt'
    scale_factor = {'Gt': 1e-6, 'Mt': 1e-3, 'kt': 1}[yaxis_scale]
    the_max = the_max_ * 1e6 * scale_factor

    extraction = pd.DataFrame([], index=[2010 + i for i in range(41)])  # 2010..2050

    # BAU baseline as a black line: historical (2010-2022) + projection (2023-2050)
    extraction.loc[2010:2022, 'BAU'] = hist_2010_2022
    extraction.loc[2023:2050, 'BAU'] = bau

    # stacked measure wedges over 2023..2050 (each = adjacent-rung difference)
    extraction.loc[2023:2050, 'CE']                     = E['CE']
    extraction.loc[2023:2050, 'Clinker substitution']   = E['SS'] - E['CE']
    extraction.loc[2023:2050, 'Sand substitution'] = E['RC'] - E['SS']
    extraction.loc[2023:2050, 'End-of-life recycling']  = E['RE'] - E['RC']
    extraction.loc[2023:2050, 'Component reuse']        = E['LE'] - E['RE']
    extraction.loc[2023:2050, 'Lifetime extension']     = E['MIU'] - E['LE']
    extraction.loc[2023:2050, 'More intensive use']     = E['BAU'] - E['MIU']

    # bridge the 2022/2023 seam: full stack height = historical value, wedges = 0
    ys = ['CE', 'Clinker substitution', 'Sand substitution', 'End-of-life recycling',
          'Component reuse', 'Lifetime extension', 'More intensive use']
    extraction.loc[2022, 'CE'] = seam
    for w in ys[1:]:
        extraction.loc[2022, w] = 0.0
    # clip sub-kt numerical noise so each stacked wedge is single-signed
    extraction[ys[1:]] = extraction[ys[1:]].clip(lower=0)

    extraction = extraction * scale_factor

    colors = ['lightgrey', 'lightblue', 'gold', 'lightgreen', 'pink', 'orange', 'steelblue']
    axx.set(xlim=(2023, 2050), ylim=(0, the_max), ylabel=f'{yaxis_scale} per year', title=country)
    extraction.plot(y='BAU', color='k', lw=1.5, legend=False, ax=axx)
    extraction.plot.area(y=ys, color=colors, lw=0, legend=False, ax=axx)
    axx.set_xticks([2023, 2030, 2040, 2050])
    axx.grid('on', ls='--', zorder=-1)
    return axx

In [6]:
# 5 pages (30 economies each) -> Supplementary Figure 8 (sand)
material = 'sand'
for j in tqdm(range(5), desc=material):
    fig = plt.figure(figsize=(10, 11), constrained_layout=True)
    plt.rcParams['font.family'] = 'Arial'
    for i, country in enumerate(economies_142[j * 30:(j + 1) * 30]):
        axx = plt.subplot(6, 5, i + 1)
        draw_ME_wedge_for_a_material(material, country, axx)
    handles, labels = axx.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center',
               bbox_to_anchor=(0.5, -.06 if j != 4 else .07),
               ncol=5, frameon=False)

    out = FIGURES / f'Supplementary-Figure-8-part{j+1}.jpg'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    plt.close(fig)

sand: 100%|██████████| 5/5 [00:28<00:00,  5.79s/it]
